# Experiment 3.18: Paradox on Diagonal Factorized Nets

This notebook is the **literate analysis front-end** for
`experiments/Tier_1_Core_Mechanism_Tests/PARADOX_diagonal_net/run_experiment.py`.
It no longer re-implements the training loop. Instead, it imports the script,
runs the same benchmark, and analyzes the returned structured results.

## Truthful scope of this first-pass benchmark

- **Question actually probed here:** does the original paradox signal persist in a
  **diagonal-factorized** deep linear control relative to a full-matrix reference?
- **What this does remove:** inter-layer **orthogonal gauge freedom** present in the
  full-matrix factorized network.
- **What this does not remove:** all reparameterization freedom. A multi-layer diagonal
  factorization still has continuous coordinate-wise rescaling symmetries and sign-related
  symmetries that preserve the end-to-end product.
- **Interpretation standard:** this notebook is a **control/null probe**, not a conclusive
  test of whether gauge symmetry is necessary for the Muon paradox.

## Notebook identity and counterpart file

- **Primary executable counterpart:** `run_experiment.py`
- **Notebook role:** reproducible exposition, tables, plots, and calibrated interpretation
- **Design rule:** the script owns the experiment core; the notebook owns explanation and analysis

This alignment matters because the previous version duplicated the implementation, which made
script/notebook drift too easy and obscured what was actually being measured.

In [ ]:
from __future__ import annotations

import importlib.util
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root(start: Path) -> Path:
    """Walk upward until the experiment script is found."""
    rel = Path("experiments/Tier_1_Core_Mechanism_Tests/PARADOX_diagonal_net/run_experiment.py")
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / rel).exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing the paradox experiment script.")


PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "Tier_1_Core_Mechanism_Tests" / "PARADOX_diagonal_net"
SCRIPT_PATH = EXPERIMENT_DIR / "run_experiment.py"

spec = importlib.util.spec_from_file_location("paradox_diagonal_factorized_experiment", SCRIPT_PATH)
experiment = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(experiment)


def format_value(value):
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if abs(float(value)) >= 1e4 or (0 < abs(float(value)) < 1e-3):
            return f"{float(value):.6e}"
        return f"{float(value):.6f}"
    return str(value)


try:
    import pandas as pd
except Exception:
    pd = None


def show_table(rows, columns=None, index=False):
    """Display a table robustly, with pandas if available and plain text otherwise."""
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.to_string(index=index))
        return df

    if columns is None:
        columns = list(rows[0].keys()) if rows else []

    col_widths = {
        col: max(len(str(col)), *(len(format_value(row.get(col, ""))) for row in rows)) if rows else len(str(col))
        for col in columns
    }
    header = " | ".join(f"{col:<{col_widths[col]}}" for col in columns)
    divider = "-+-".join("-" * col_widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(f"{format_value(row.get(col, '')):<{col_widths[col]}}" for col in columns))
    return rows


print(f"Project root:   {PROJECT_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Script path:    {SCRIPT_PATH}")
print("Notebook import status: OK")

## Reproducibility, runtime, and exact benchmark identity

This cell executes the **default script benchmark** without changing the scientific setup.
The defaults are intentionally preserved from the earlier pair as closely as possible:

- dimension `32`
- depth `4`
- `20` independent runs per optimizer
- `500` training steps per run
- one fixed train batch and one fixed finite test set
- separate learning-rate searches for diagonal and full-matrix settings

The script now returns structured raw results, so the notebook can present the exact same run
without copying the implementation.

In [ ]:
notebook_t0 = time.time()
results = experiment.run_experiment(emit_summary=False)
notebook_runtime = time.time() - notebook_t0

config = results["config"]
seed_start = config["init_seed_base"]
seed_end = seed_start + config["num_independent_runs"] - 1

print("Benchmark execution complete.")
print(f"Script-reported runtime:   {results['runtime_seconds']:.2f} s")
print(f"Notebook cell wall-clock:  {notebook_runtime:.2f} s")
print()
print("Configuration")
for key in [
    "global_seed",
    "dim",
    "num_layers",
    "num_steps",
    "batch_size",
    "momentum",
    "ns_iters",
    "num_independent_runs",
    "num_test_inputs",
    "data_scale",
    "warmup_steps",
]:
    print(f"  {key:>22}: {config[key]}")
print(f"  {'run_seed_range':>22}: {seed_start}..{seed_end}")
print(f"  {'diag_lr_candidates':>22}: {config['diag_lr_candidates']}")
print(f"  {'full_lr_candidates':>22}: {config['full_lr_candidates']}")

## Implemented metrics and what they mean

The benchmark now distinguishes three different notions of diversity:

1. **Weight diversity**
   - pairwise distance between final parameter tensors across runs
   - diagonal case: Euclidean distance across all learned layer diagonals
   - full case: Euclidean/Frobenius distance across all learned layer matrices

2. **Sampled output diversity**
   - pairwise Frobenius distance between final outputs on a fixed finite test set `X_test`
   - normalized by `||X_test||_F`
   - this is the same quantity previously called “function diversity”
   - it is a useful surrogate, but it is not an exact operator comparison by itself

3. **Exact end-to-end operator diversity**
   - diagonal case: pairwise distance between the learned products `d_L ⊙ ... ⊙ d_1`
   - full case: pairwise Frobenius distance between the learned end-to-end matrices `W_L ... W_1`

The script also retains the original **heuristic paradox rules**. These remain useful benchmark
checks, but they should not be described as formal statistical hypothesis tests.

## Learning-rate search summary

The benchmark preserves separate learning-rate searches for the diagonal-factorized control and the
full-matrix reference. The table below shows the trial records and the selected first stable rate
for each optimizer/architecture pair.

In [ ]:
lr_rows = []
for arch_key, arch_label in [("diagonal", "Diagonal factorized"), ("full_matrix", "Full matrix")]:
    for opt_name in ["SGD", "Muon"]:
        search = results["learning_rates"][arch_key][opt_name]
        for trial in search["trials"]:
            lr_rows.append(
                {
                    "architecture": arch_label,
                    "optimizer": opt_name,
                    "candidate_lr": trial["lr"],
                    "stable": trial["stable"],
                    "steps_completed": trial["steps_completed"],
                    "initial_loss": trial["initial_loss"],
                    "final_loss": trial["final_loss"],
                    "selected_lr": search["selected_lr"],
                }
            )

show_table(
    lr_rows,
    columns=[
        "architecture",
        "optimizer",
        "candidate_lr",
        "stable",
        "steps_completed",
        "initial_loss",
        "final_loss",
        "selected_lr",
    ],
)

### LR interpretation

This is still a lightweight stability screen rather than a full tuning study. It is enough for the
current benchmark because the goal is **matched protocol continuity**, not an aggressively optimized
comparison. The important point is that the notebook is now explicit about the candidate grids,
trial outcomes, and selected rates.

## Main summary table: losses and diversity statistics

The next table aggregates the returned raw results into one row per architecture/optimizer pair.
The key ratios are:

- **Sampled-output / weight**: the original paradox-style summary
- **Operator / weight**: a stronger exact end-to-end counterpart

In [ ]:
summary_rows = results["summary_rows"]
show_table(
    summary_rows,
    columns=[
        "architecture",
        "optimizer",
        "lr",
        "loss_mean",
        "loss_std",
        "weight_diversity_mean",
        "sampled_output_diversity_mean",
        "operator_diversity_mean",
        "sampled_output_over_weight",
        "operator_over_weight",
        "num_diverged_runs",
    ],
)

## Loss trajectories across runs

Because the script now returns per-run loss histories, we can inspect the optimization trajectories
instead of only the endpoint means. The plots below show the mean and one standard deviation across
independent runs for each optimizer, separately for the diagonal-factorized control and the full
reference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)

for ax, arch_key, arch_label in [
    (axes[0], "diagonal", "Diagonal factorized"),
    (axes[1], "full_matrix", "Full matrix"),
]:
    for opt_name, color in [("SGD", "tab:blue"), ("Muon", "tab:orange")]:
        histories = results[arch_key]["optimizers"][opt_name]["loss_histories"]
        mean_curve = np.nanmean(histories, axis=0)
        std_curve = np.nanstd(histories, axis=0)
        steps = np.arange(mean_curve.shape[0])
        ax.plot(steps, mean_curve, label=opt_name, color=color, linewidth=2)
        ax.fill_between(steps, mean_curve - std_curve, mean_curve + std_curve, color=color, alpha=0.2)
    ax.set_title(arch_label)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Training loss")
    ax.set_yscale("log")
    ax.legend()

fig.suptitle("Loss trajectories: mean ± 1 std across independent runs", y=1.03)
fig.tight_layout()
plt.show()

### Interim reading of the loss curves

A serious benchmark should separate at least three questions:

1. Do both optimizers remain stably trainable under the selected rates?
2. Do they reach comparable or systematically different final losses?
3. Are diversity effects appearing only in parameter space, or also in the exact end-to-end operator?

The endpoint tables below address (2) and (3) more directly.

## Pairwise diversity scatter: weight distance vs sampled output distance

The original paradox story is about **large parameter differences with relatively small behavioral
changes**. The scatter plots below expose that geometry directly for all run pairs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=False, sharey=False)
plot_specs = [
    (0, 0, "diagonal", "SGD", "Diagonal factorized / SGD"),
    (0, 1, "diagonal", "Muon", "Diagonal factorized / Muon"),
    (1, 0, "full_matrix", "SGD", "Full matrix / SGD"),
    (1, 1, "full_matrix", "Muon", "Full matrix / Muon"),
]

for row, col, arch_key, opt_name, title in plot_specs:
    ax = axes[row, col]
    opt_results = results[arch_key]["optimizers"][opt_name]
    x = opt_results["pairwise_weight_distances"]
    y = opt_results["pairwise_sampled_output_distances"]
    ax.scatter(x, y, s=28, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("Pairwise weight distance")
    ax.set_ylabel("Pairwise sampled output distance")

fig.suptitle("Pairwise diversity geometry across independent runs", y=1.02)
fig.tight_layout()
plt.show()

## Exact operator diversity summary

The previous pair discussed “same function” more strongly than the implementation supported. The
script now exposes an exact end-to-end operator metric from the trained weights, so the notebook can
separate:

- sampled output agreement on a finite test set, and
- exact operator agreement of the learned linear map.

In [ ]:
operator_rows = []
for arch_key, arch_label in [("diagonal", "Diagonal factorized"), ("full_matrix", "Full matrix")]:
    arch_summary = results[arch_key]["summary"]
    for opt_name in ["SGD", "Muon"]:
        opt_results = results[arch_key]["optimizers"][opt_name]
        operator_rows.append(
            {
                "architecture": arch_label,
                "optimizer": opt_name,
                "operator_diversity_mean": opt_results["operator_diversity_mean"],
                "operator_diversity_std": opt_results["operator_diversity_std"],
                "operator_over_weight": arch_summary["operator_ratio_sgd"] if opt_name == "SGD" else arch_summary["operator_ratio_muon"],
                "operator_paradox_strength": arch_summary["operator_paradox_strength"],
            }
        )

show_table(
    operator_rows,
    columns=[
        "architecture",
        "optimizer",
        "operator_diversity_mean",
        "operator_diversity_std",
        "operator_over_weight",
        "operator_paradox_strength",
    ],
)

## Explicit caveats for this control

This benchmark is intentionally modest. Its main caveats are now stated plainly:

1. **Diagonal factorization is not the same as a truly non-factorized gauge-free model.**
   A 4-layer diagonal product still admits continuous re-scaling symmetries across layers.

2. **The full reference differs in more than one way.**
   It changes parameter count, geometry, gradient structure, and effective optimization landscape,
   not just the presence of orthogonal gauge freedom.

3. **Sampled output diversity is not exact functional equality.**
   The notebook now reports exact operator diversity too, but the main heuristic benchmark still
   uses the original sampled-output-over-weight ratio for continuity.

4. **The heuristic checks are not formal inference.**
   There are no confidence intervals, bootstrap tests, or repeated-dataset uncertainty estimates in
   this first pass.

## Heuristic outcome and calibrated conclusion

The next cell prints the retained heuristic checks together with a narrow conclusion. The language is
now aligned with what the benchmark actually measures.

In [ ]:
heuristics = results["heuristics"]

heuristic_rows = []
for key, test in heuristics["tests"].items():
    row = {
        "heuristic": key,
        "passed": test["passed"],
        "description": test["description"],
    }
    for obs_key, obs_value in test["observed"].items():
        row[obs_key] = obs_value
    heuristic_rows.append(row)

show_table(heuristic_rows)

print()
print(f"Heuristic tally: {heuristics['total_pass']}/4")
print(f"Verdict label:   {heuristics['verdict_label']}")
print(f"Verdict message: {heuristics['verdict_message']}")
print()
print("Calibrated reading:")
print(
    f"  Diagonal-factorized sampled-output paradox strength: {heuristics['diagonal_sampled_output_paradox_strength']:.4f}"
)
print(
    f"  Full-matrix sampled-output paradox strength:         {heuristics['full_sampled_output_paradox_strength']:.4f}"
)
print()
print(
    "  In this first-pass control, the diagonal-factorized model does not cleanly eliminate the paradox signal."
)
print(
    "  That weakens any claim that the benchmark, as implemented here, demonstrates orthogonal gauge freedom"
)
print(
    "  is necessary for the observed Muon paradox. The result should be treated as a careful control outcome,"
)
print(
    "  not as a conclusive negative or positive theorem about gauge mechanisms."
)

## Recommended next verification step

A stronger follow-up would keep this notebook/script alignment but add formal uncertainty estimates
(e.g. bootstrap confidence intervals over pairwise summaries) and, if the scientific question demands
it, introduce a **truly non-factorized diagonal control** in which the end-to-end diagonal operator is
parameterized directly rather than as a deep product.